# Multi-Region LLM Agent Infrastructure — Experiment Analysis
IEEE TKDE DK-GenAI Special Issue

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.2)
RESULTS_DIR = Path('../results')
FIGURES_DIR = Path('../figures')
FIGURES_DIR.mkdir(exist_ok=True)

## Load Experiment Results

In [ ]:
exp_a = pd.read_csv(RESULTS_DIR / 'experiment_a/experiment_a_raw.csv')
exp_b = pd.read_csv(RESULTS_DIR / 'experiment_b/experiment_b_raw.csv')
exp_c = pd.read_csv(RESULTS_DIR / 'experiment_c/experiment_c_raw.csv')

print('Exp A:', exp_a.shape)
print('Exp B:', exp_b.shape)
print('Exp C:', exp_c.shape)

## Figure 1 — Experiment A: Write Engine Latency CDF

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

WRITE_ALGO_LABELS = {
    'W0': 'W0: Baseline (naive)',
    'W1': 'W1: Selective Flush',
    'W2': 'W2: WAL + Async Batch',
    'W3': 'W3: CRDT Merge',
    'W4': 'W4: Adaptive Pre-flush',
}

colors = sns.color_palette('tab10', n_colors=5)

for ax, metric, title in zip(
    axes,
    ['write_latency_ms', 'flush_latency_ms'],
    ['Per-trace Write Latency (ms)', 'Flush Latency (ms)'],
):
    for i, algo in enumerate(['W0', 'W1', 'W2', 'W3', 'W4']):
        subset = exp_a[exp_a['write_algorithm'] == algo][metric].dropna()
        if subset.empty:
            continue
        sorted_vals = np.sort(subset)
        cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
        ax.plot(sorted_vals, cdf, label=WRITE_ALGO_LABELS[algo], color=colors[i], lw=2)
    ax.set_xlabel(title)
    ax.set_ylabel('CDF')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_xlim(left=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_write_engine_cdf.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig1_write_engine_cdf.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2 — Experiment A: Write Latency Box Plot (p50/p95/p99)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = ['W0', 'W1', 'W2', 'W3', 'W4']
sns.boxplot(
    data=exp_a,
    x='write_algorithm', y='write_latency_ms',
    order=order, palette='tab10', ax=ax,
    showfliers=False,
)
ax.set_xlabel('Write Algorithm')
ax.set_ylabel('Write Latency (ms)')
ax.set_title('Experiment A: Write Engine Comparison')
ax.set_xticklabels([WRITE_ALGO_LABELS.get(a, a) for a in order], rotation=15, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_write_boxplot.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig2_write_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3 — Experiment B: Read Engine Handoff Latency

In [ ]:
READ_ALGO_LABELS = {
    'R0': 'R0: Full Dump (baseline)',
    'R1': 'R1: Hydration Protocol',
    'R2': 'R2: LLM Summarization',
    'R3': 'R3: Semantic RAG',
    'R4': 'R4: MemGPT Hierarchical',
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics_titles = [
    ('handoff_latency_ms', 'Handoff Latency (ms)'),
    ('context_token_count', 'Context Tokens (input)'),
    ('state_integrity_score', 'State Integrity Score'),
]

order_r = ['R0', 'R1', 'R2', 'R3', 'R4']
for ax, (metric, title) in zip(axes, metrics_titles):
    sns.boxplot(
        data=exp_b, x='read_algorithm', y=metric,
        order=order_r, palette='tab10', ax=ax, showfliers=False,
    )
    ax.set_xlabel('Read Algorithm')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xticklabels([READ_ALGO_LABELS.get(a, a) for a in order_r], rotation=20, ha='right', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_read_engine_comparison.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig3_read_engine_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 4 — Experiment B: Compression Ratio vs Integrity Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors_r = sns.color_palette('tab10', 5)

for i, algo in enumerate(['R0', 'R1', 'R2', 'R3', 'R4']):
    subset = exp_b[exp_b['read_algorithm'] == algo]
    ax.scatter(
        subset['compression_ratio'],
        subset['state_integrity_score'],
        label=READ_ALGO_LABELS.get(algo, algo),
        alpha=0.5, color=colors_r[i], s=30,
    )

ax.set_xlabel('Compression Ratio (higher = smaller payload)')
ax.set_ylabel('State Integrity Score')
ax.set_title('Read Engine: Compression vs Integrity Trade-off')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_compression_vs_integrity.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig4_compression_vs_integrity.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 5 — Experiment C: Hybrid vs All Conditions (Grouped Bar)

In [ ]:
summary_c = exp_c.groupby('condition').agg(
    write_p50=('write_latency_ms', lambda x: np.percentile(x, 50)),
    handoff_p50=('handoff_latency_ms', lambda x: np.percentile(x, 50)),
    cost_mean=('estimated_cost_usd', 'mean'),
    tokens_mean=('context_token_count', 'mean'),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
conditions = summary_c['condition'].tolist()
x = np.arange(len(conditions))
w = 0.35

axes[0].bar(x, summary_c['write_p50'], width=w, label='Write p50 (ms)', color='steelblue')
axes[0].bar(x + w, summary_c['handoff_p50'], width=w, label='Handoff p50 (ms)', color='coral')
axes[0].set_xticks(x + w / 2)
axes[0].set_xticklabels(conditions, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Experiment C: Latency by Condition')
axes[0].legend()

axes[1].bar(x, summary_c['cost_mean'] * 1000, width=0.6, color='mediumseagreen')
axes[1].set_xticks(x)
axes[1].set_xticklabels(conditions, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('Avg Cost per Iteration (millicents)')
axes[1].set_title('Experiment C: Cost by Condition')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_hybrid_comparison.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig5_hybrid_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Table 1 — Statistical Significance (Wilcoxon Tests)

In [ ]:
wtest_files = sorted(Path('../results/experiment_c').glob('wilcoxon_*.csv'))
if wtest_files:
    all_tests = pd.concat([pd.read_csv(f) for f in wtest_files], ignore_index=True)
    display_cols = ['metric', 'comparison', 'n', 'p_value', 'significant_p05',
                    'effect_size', 'median_reduction']
    print(all_tests[display_cols].to_string(index=False))
else:
    print('No Wilcoxon test files found. Run experiments first.')

## Cost Summary

In [ ]:
all_data = pd.concat([exp_a, exp_b, exp_c], ignore_index=True)
total_cost = all_data['estimated_cost_usd'].sum()
total_iters = len(all_data)
print(f'Total iterations across all experiments: {total_iters}')
print(f'Total estimated API cost: ${total_cost:.4f}')
print(f'Average cost per iteration: ${total_cost/max(total_iters,1):.6f}')